# Hybrid KD: INT8 Full Validation Quality Gate

Đánh giá FP32 và ứng viên INT8 trên toàn bộ 5.000 ảnh Karpathy Validation.
Gate giữ nguyên: CIDEr-D ≥ −0,02; BLEU-4 ≥ −0,01; CE ≤ +0,05.

## Input bắt buộc — đúng 5 input

1. Output `HYB_INT8_Preflight`;
2. output `01_MSCOCO_Data_Contract`;
3. dataset `mscoco2014-1`;
4. dataset/package `pycocoevalcap-1.2`;
5. dataset `ONNX_Runtime_Python313_Offline`.

Khi resume, gắn thêm output gần nhất của notebook này.

Runtime: `CPU / Accelerator None / Internet Off`.

## Cell 1 — Authorization, integrity và khóa protocol

In [1]:
import csv
import hashlib
import importlib
import importlib.util
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

STARTED = time.perf_counter()
PAIR = "HybridKD-S1"
PAIR_SLUG = "hybrid_kd_s1_seed2026"
STUDENT_ID = "S1"
SEED = 2026
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = INPUT_ROOT / "notebooks" if (INPUT_ROOT / "notebooks").is_dir() else INPUT_ROOT
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working/HYB_INT8_Full_Validation",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
PREDICTION = OUT / "predictions"
MANIFEST = OUT / "manifests"
DEPENDENCY = Path(os.environ.get(
    "KLCN_DEPENDENCY_ROOT", "/tmp/klcn_hyb_int8_quality_dependencies"
))
for directory in [OUT, AUDIT, CONFIG, RESULT, PREDICTION, MANIFEST, DEPENDENCY]:
    directory.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.set_grad_enabled(False)
torch.manual_seed(42)
np.random.seed(42)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(path for path in NOTEBOOK_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, f"Cần đúng 1 file {name!r}; tìm thấy {len(matches)}: {matches[:10]}"
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents
    return target

PREFLIGHT_PATH = exactly_one(
    "HYB_INT8_PREFLIGHT_COMPLETE.json"
)
AUTH_PATH = exactly_one(
    "HYB_INT8_FULL_VALIDATION_AUTHORIZED.json"
)
PREFLIGHT_RESULT_PATH = exactly_one("int8_validation_preflight_result.json")
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")

PREFLIGHT = read_json(PREFLIGHT_PATH)
AUTH = read_json(AUTH_PATH)
PREFLIGHT_RESULT = read_json(PREFLIGHT_RESULT_PATH)
P1 = read_json(P1_PATH)
PREFLIGHT_ROOT = PREFLIGHT_PATH.parent
P7G_ROOT = PREFLIGHT_ROOT  # Alias cho logic quality gate đã kiểm chứng.

assert PREFLIGHT["status"] == "complete"
assert PREFLIGHT["decision"] == "int8_candidate_selected"
assert PREFLIGHT["selected_candidate"] == "dynamic_weight_int8"
assert int(PREFLIGHT["selected_seed"]) == SEED
assert PREFLIGHT["full_validation_quality_authorized"] is True
assert PREFLIGHT["quality_claim_made"] is False
assert PREFLIGHT["physical_android_benchmark_performed"] is False
assert PREFLIGHT["test_set_status"] == "sealed" and PREFLIGHT["test_set_used"] is False
assert AUTH["status"] == "authorized"
assert AUTH["selected_candidate"] == "dynamic_weight_int8"
assert int(AUTH["selected_seed"]) == SEED
assert AUTH["constraints"] == {
    "same_5000_validation_images": True,
    "same_beam_size": 3,
    "same_length_penalty": 0.7,
    "new_training": False,
    "checkpoint_reselection": False,
    "test_opening": False,
    "android_benchmark_before_quality_gate": False,
}
assert PREFLIGHT_RESULT["status"] == "passed"
assert PREFLIGHT_RESULT["decision"] == "int8_candidate_selected"
assert PREFLIGHT_RESULT["selected_candidate"]["name"] == "dynamic_weight_int8"
assert PREFLIGHT_RESULT["selected_candidate"]["eligible"] is True
assert P1["status"] == "ready"

manifest_path = resolve_inside(PREFLIGHT_ROOT, PREFLIGHT["manifest"]["relative_path"])
assert sha256(manifest_path) == PREFLIGHT["manifest"]["sha256"]
with manifest_path.open("r", encoding="utf-8", newline="") as handle:
    preflight_manifest_rows = list(csv.DictReader(handle))
assert len(preflight_manifest_rows) == int(PREFLIGHT["manifest"]["rows"])
for row in preflight_manifest_rows:
    artifact = resolve_inside(PREFLIGHT_ROOT, row["relative_path"])
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["bytes"])
    assert sha256(artifact) == row["sha256"]

artifact_map = {}
role_map = {
    "encoder_fp32": "encoder_fp32",
    "decoder_fp32": "decoder_fp32",
    "encoder_candidate": "encoder_int8",
    "decoder_candidate": "decoder_int8",
}
for role, auth_role in role_map.items():
    reference = AUTH["artifacts"][auth_role]
    artifact = resolve_inside(PREFLIGHT_ROOT, reference["relative_path"])
    assert artifact.is_file() and sha256(artifact) == reference["sha256"]
    artifact_map[role] = {
        "relative_path": reference["relative_path"],
        "size_bytes": artifact.stat().st_size,
        "sha256": reference["sha256"],
    }

selected_preflight = PREFLIGHT_RESULT["selected_candidate"]
READY = {
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "strategy": "dynamic_weight_int8",
    "artifacts": artifact_map,
    "compatibility_result": {
        "preflight_passed": True,
        "gates": selected_preflight["gates"],
        "caption_quality_evaluated": False,
    },
}
SELECTED = READY["compatibility_result"]
assert SELECTED["preflight_passed"] is True and all(SELECTED["gates"].values())
QUALITY_GATES = dict(AUTH["quality_gates"])
assert QUALITY_GATES == {
    "CIDEr-D_minimum_delta": -0.02,
    "BLEU-4_minimum_delta": -0.01,
    "CE_maximum_delta": 0.05,
    "finite_required": True,
}

DEPLOYMENT_CONTRACT_PATH = PREFLIGHT_ROOT / "package/fp32_deployment_contract.json"
VOCAB_PATH = PREFLIGHT_ROOT / "package/word_vocabulary_minfreq5.json"
assert DEPLOYMENT_CONTRACT_PATH.is_file() and VOCAB_PATH.is_file()

write_json(AUDIT / "cell01_entry_integrity.json", {
    "schema_version": "hybrid_kd_int8_full_quality_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "student_id": STUDENT_ID,
    "seed": SEED,
    "selected_candidate": READY["strategy"],
    "preflight_manifest_rows": len(preflight_manifest_rows),
    "quality_gates": QUALITY_GATES,
    "runtime": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cpu_threads": torch.get_num_threads(),
    },
    "new_training": False,
    "checkpoint_reselection": False,
    "new_test_opening": False,
    "test_data_read_this_phase": False,
    "prior_test_open_count": 0,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("CELL 1 — ENTRY INTEGRITY")
print("Status                  : PASSED")
print("Student / seed          :", STUDENT_ID, "/", SEED)
print("INT8 candidate          :", READY["strategy"])
print("Preflight manifest      :", len(preflight_manifest_rows), "/", len(preflight_manifest_rows), "VERIFIED")
print("New training            : NOT STARTED")
print("Held-out Test reopened  : False")
print("Progress                : 1 / 6 code cells")

CELL 1 — ENTRY INTEGRITY
Status                  : PASSED
Student / seed          : S1 / 2026
INT8 candidate          : dynamic_weight_int8
Preflight manifest      : 20 / 20 VERIFIED
New training            : NOT STARTED
Held-out Test reopened  : False
Progress                : 1 / 6 code cells


## Cell 2 — Validation corpus, vocabulary, preprocessing và COCO metrics

In [2]:
STARTED = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

P1_ROOT = P1_PATH.parent
VAL_PARQUET = P1_ROOT / "parquet/mscoco_validation.parquet"
assert VAL_PARQUET.is_file() and VOCAB_PATH.is_file()

table = pq.read_table(
    VAL_PARQUET,
    columns=[
        "annotation_id", "image_id", "karpathy_split",
        "source_partition", "caption", "token_ids",
    ],
)
VALIDATION_REFERENCES = defaultdict(list)
VALIDATION_TOKEN_ROWS = defaultdict(list)
for row in table.to_pylist():
    assert row["karpathy_split"] == "val"
    assert row["source_partition"] == "val2014"
    image_id = int(row["image_id"])
    VALIDATION_REFERENCES[image_id].append(str(row["caption"]))
    VALIDATION_TOKEN_ROWS[image_id].append((
        int(row["annotation_id"]),
        tuple(int(value) for value in row["token_ids"]),
    ))
for image_id in VALIDATION_TOKEN_ROWS:
    VALIDATION_TOKEN_ROWS[image_id].sort()
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_REFERENCES))
assert len(VALIDATION_IMAGE_IDS) == 5000
assert sum(map(len, VALIDATION_REFERENCES.values())) == 25010

vocabulary = read_json(VOCAB_PATH)
raw_id_to_token = vocabulary["id_to_token"]
ID_TO_TOKEN = {
    index: token for index, token in enumerate(raw_id_to_token)
}
assert len(ID_TO_TOKEN) == 10080
assert [ID_TO_TOKEN[index] for index in range(4)] == [
    "<pad>", "<bos>", "<eos>", "<unk>",
]

def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)

deployment_contract = read_json(DEPLOYMENT_CONTRACT_PATH)
preprocess_contract = deployment_contract["preprocessing"]
assert preprocess_contract["layout"] == "NCHW"
assert preprocess_contract["dtype"] == "float32"
crop = preprocess_contract["center_crop"]
assert len(crop) == 2 and crop[0] == crop[1] == 224
VALIDATION_TRANSFORM = transforms.Compose([
    transforms.Resize(
        int(preprocess_contract["resize_shortest_side"]),
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    ),
    transforms.CenterCrop(int(crop[0])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=preprocess_contract["mean"],
        std=preprocess_contract["std"],
    ),
])

known_val_roots = [
    INPUT_ROOT / "datasets/nguynthnhdan/mscoco2014-1/val2014",
    INPUT_ROOT / "datasets/nguynthnhdan/mscoco2014-1/val2014/val2014",
    INPUT_ROOT / "mscoco2014-1/val2014",
    INPUT_ROOT / "mscoco2014-1/val2014/val2014",
]
candidate_roots = {
    path.resolve() for path in known_val_roots if path.is_dir()
}
for pattern in ["*/val2014", "*/*/val2014", "*/*/*/val2014", "*/*/*/*/val2014"]:
    for path in INPUT_ROOT.glob(pattern):
        if path.is_dir():
            candidate_roots.add(path.resolve())
            if (path / "val2014").is_dir():
                candidate_roots.add((path / "val2014").resolve())

def filename(image_id):
    return f"COCO_val2014_{int(image_id):012d}.jpg"

probe_ids = VALIDATION_IMAGE_IDS[:16]
valid_roots = [
    root for root in sorted(candidate_roots)
    if all((root / filename(image_id)).is_file() for image_id in probe_ids)
]
if not valid_roots:
    dataset_root = INPUT_ROOT / "datasets"
    search_root = dataset_root if dataset_root.is_dir() else INPUT_ROOT
    recursive = {
        path.resolve() for path in search_root.rglob("val2014") if path.is_dir()
    }
    recursive |= {
        (path / "val2014").resolve()
        for path in recursive if (path / "val2014").is_dir()
    }
    valid_roots = [
        root for root in sorted(recursive)
        if all((root / filename(image_id)).is_file() for image_id in probe_ids)
    ]
assert len(valid_roots) == 1, valid_roots
VAL_ROOT = valid_roots[0]

try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(tokenized_gt, tokenized_pred)
    meteor, _ = Meteor().compute_score(tokenized_gt, tokenized_pred)
    rouge, _ = Rouge().compute_score(tokenized_gt, tokenized_pred)
    cider, _ = Cider().compute_score(tokenized_gt, tokenized_pred)
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }

smoke_refs = {
    image_id: VALIDATION_REFERENCES[image_id]
    for image_id in VALIDATION_IMAGE_IDS[:16]
}
smoke_hyps = {
    image_id: VALIDATION_REFERENCES[image_id][0]
    for image_id in VALIDATION_IMAGE_IDS[:16]
}
smoke_metrics = compute_coco_metrics(smoke_refs, smoke_hyps)
assert all(math.isfinite(value) for value in smoke_metrics.values())

write_json(AUDIT / "cell02_validation_data.json", {
    "schema_version": "hybrid_kd_int8_full_quality_validation_data_v1",
    "created_at_utc": now(),
    "status": "passed",
    "validation_images": len(VALIDATION_IMAGE_IDS),
    "validation_references": sum(map(len, VALIDATION_REFERENCES.values())),
    "vocabulary_size": len(ID_TO_TOKEN),
    "metrics_smoke": smoke_metrics,
    "caption_text_read": True,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_data_read_this_phase": False,
    "new_test_opening": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("CELL 2 — VALIDATION DATA")
print("Status                  : PASSED")
print("Validation images       : 5,000")
print("Validation references   : 25,010")
print("Vocabulary              : 10,080")
print("COCO metrics smoke      : PASSED")
print("Held-out Test           : SEALED / NOT READ")
print("Progress                : 2 / 6 code cells")

Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 997 tokens at 8891.62 tokens per second.
PTBTokenizer tokenized 200 tokens at 2432.30 tokens per second.


{'testlen': 171, 'reflen': 171, 'guess': [171, 155, 139, 123], 'correct': [171, 155, 139, 123]}
ratio: 0.9999999999941521
CELL 2 — VALIDATION DATA
Status                  : PASSED
Validation images       : 5,000
Validation references   : 25,010
Vocabulary              : 10,080
COCO metrics smoke      : PASSED
Held-out Test           : SEALED / NOT READ
Progress                : 2 / 6 code cells


## Cell 3 — Resolve FP32/INT8 artifacts và ONNX Runtime smoke

In [3]:
STARTED = time.perf_counter()

def ensure_onnxruntime():
    from importlib import metadata as importlib_metadata
    expected_version = "1.28.0"
    if importlib.util.find_spec("onnxruntime") is not None:
        installed_version = importlib_metadata.version("onnxruntime")
        if installed_version == expected_version:
            return {"status": "preinstalled", "version": installed_version, "source": None}

    python_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    ort_wheels = sorted(INPUT_ROOT.rglob(
        f"onnxruntime-{expected_version}-{python_tag}-{python_tag}-*.whl"
    ))
    assert len(ort_wheels) == 1, (
        "Cần đúng một ONNX Runtime wheel tương thích Python hiện tại; "
        f"tìm thấy {len(ort_wheels)}: {ort_wheels}"
    )
    wheel_dir = ort_wheels[0].parent
    dependencies = []
    for pattern in ("flatbuffers-*.whl", "packaging-*.whl", "protobuf-*.whl"):
        matches = sorted(wheel_dir.glob(pattern))
        assert len(matches) == 1, (pattern, matches)
        dependencies.append(matches[0])
    command = [
        sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
        "--force-reinstall",
        *[str(path) for path in dependencies], str(ort_wheels[0]),
    ]
    completed = subprocess.run(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, check=False,
    )
    install_log = AUDIT / "onnxruntime_offline_install.log"
    install_log.write_text(completed.stdout or "", encoding="utf-8")
    assert completed.returncode == 0, f"Cài ORT offline thất bại; xem {install_log}"
    importlib.invalidate_caches()
    assert importlib.util.find_spec("onnxruntime") is not None
    assert importlib_metadata.version("onnxruntime") == expected_version
    return {"status": "installed_offline", "version": expected_version, "source": str(ort_wheels[0])}

ORT_DEPENDENCY = ensure_onnxruntime()
import onnxruntime as ort

assert "CPUExecutionProvider" in ort.get_available_providers()
artifact_map = READY["artifacts"]
FP32_ENCODER_PATH = resolve_inside(
    P7G_ROOT, artifact_map["encoder_fp32"]["relative_path"]
)
FP32_DECODER_PATH = resolve_inside(
    P7G_ROOT, artifact_map["decoder_fp32"]["relative_path"]
)
INT8_ENCODER_PATH = resolve_inside(
    P7G_ROOT, artifact_map["encoder_candidate"]["relative_path"]
)
INT8_DECODER_PATH = resolve_inside(
    P7G_ROOT, artifact_map["decoder_candidate"]["relative_path"]
)
for role, path in {
    "encoder_fp32": FP32_ENCODER_PATH,
    "decoder_fp32": FP32_DECODER_PATH,
    "encoder_candidate": INT8_ENCODER_PATH,
    "decoder_candidate": INT8_DECODER_PATH,
}.items():
    reference = artifact_map[role]
    assert path.is_file()
    assert path.stat().st_size == int(reference["size_bytes"])
    assert sha256(path) == reference["sha256"]

session_options = ort.SessionOptions()
session_options.intra_op_num_threads = 1
session_options.inter_op_num_threads = 1
session_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

def ort_session(path):
    return ort.InferenceSession(
        str(path),
        sess_options=session_options,
        providers=["CPUExecutionProvider"],
    )

class ORTEncoder:
    def __init__(self, path):
        self.session = ort_session(path)
        assert [item.name for item in self.session.get_inputs()] == ["image"]
        assert [item.name for item in self.session.get_outputs()] == ["visual_memory"]

    def __call__(self, image):
        array = np.ascontiguousarray(
            image.detach().cpu().numpy().astype(np.float32, copy=False)
        )
        output = self.session.run(None, {"image": array})[0]
        return torch.from_numpy(np.asarray(output))

class ORTDecoder:
    """Adapter cho decoder ONNX batch cố định 3.

    Mọi batch 1..N được chia thành các nhóm tối đa 3, đệm đến đúng batch 3,
    chạy ORT rồi cắt phần đệm. Điều này giữ đúng graph đã parity ở preflight.
    """
    FIXED_BATCH = 3

    def __init__(self, path):
        self.session = ort_session(path)
        assert [item.name for item in self.session.get_inputs()] == [
            "decoder_input_ids", "visual_memory", "decoder_attention_mask"
        ]
        assert [item.name for item in self.session.get_outputs()] == ["logits"]

    def __call__(self, token_ids, memory, attention_mask):
        assert token_ids.ndim == 2 and token_ids.shape[1] == 31
        assert memory.ndim == 3 and tuple(memory.shape[1:]) == (49, 256)
        assert attention_mask.shape == token_ids.shape
        assert token_ids.shape[0] == memory.shape[0] == attention_mask.shape[0]
        outputs = []
        for offset in range(0, token_ids.shape[0], self.FIXED_BATCH):
            ids = token_ids[offset:offset + self.FIXED_BATCH]
            mem = memory[offset:offset + self.FIXED_BATCH]
            mask = attention_mask[offset:offset + self.FIXED_BATCH]
            actual = ids.shape[0]
            if actual < self.FIXED_BATCH:
                pad = self.FIXED_BATCH - actual
                ids = torch.cat([ids, torch.zeros((pad, 31), dtype=ids.dtype)], dim=0)
                mem = torch.cat([mem, mem[-1:].expand(pad, -1, -1)], dim=0)
                mask = torch.cat([mask, torch.zeros((pad, 31), dtype=mask.dtype)], dim=0)
            feeds = {
                "decoder_input_ids": np.ascontiguousarray(ids.cpu().numpy().astype(np.int64, copy=False)),
                "visual_memory": np.ascontiguousarray(mem.cpu().numpy().astype(np.float32, copy=False)),
                "decoder_attention_mask": np.ascontiguousarray(mask.cpu().numpy().astype(np.int64, copy=False)),
            }
            output = self.session.run(None, feeds)[0]
            outputs.append(torch.from_numpy(np.asarray(output[:actual])))
        return torch.cat(outputs, dim=0)

FP32_ENCODER = ORTEncoder(FP32_ENCODER_PATH)
FP32_DECODER = ORTDecoder(FP32_DECODER_PATH)
INT8_ENCODER = ORTEncoder(INT8_ENCODER_PATH)
INT8_DECODER = ORTDecoder(INT8_DECODER_PATH)

smoke_id = VALIDATION_IMAGE_IDS[0]
with Image.open(VAL_ROOT / filename(smoke_id)) as image_handle:
    smoke_image = VALIDATION_TRANSFORM(
        image_handle.convert("RGB")
    ).unsqueeze(0)
smoke_ids = torch.zeros((3, 31), dtype=torch.long)
smoke_ids[:, 0] = 1
smoke_mask = smoke_ids.ne(0)
fp32_memory = FP32_ENCODER(smoke_image)
int8_memory = INT8_ENCODER(smoke_image)
BATCHED_DECODER_SUPPORTED = True
BATCHED_DECODER_ERROR = None
try:
    fp32_batch_logits = FP32_DECODER(
        smoke_ids,
        fp32_memory.expand(3, -1, -1).contiguous(),
        smoke_mask,
    )
    int8_batch_logits = INT8_DECODER(
        smoke_ids,
        int8_memory.expand(3, -1, -1).contiguous(),
        smoke_mask,
    )
    assert tuple(fp32_batch_logits.shape) == (3, 31, 10080)
    assert tuple(int8_batch_logits.shape) == (3, 31, 10080)
    assert torch.isfinite(fp32_batch_logits).all()
    assert torch.isfinite(int8_batch_logits).all()
except Exception as error:
    BATCHED_DECODER_SUPPORTED = False
    BATCHED_DECODER_ERROR = f"{type(error).__name__}: {error}"
    fp32_single_logits = FP32_DECODER(
        smoke_ids[:1], fp32_memory, smoke_mask[:1]
    )
    int8_single_logits = INT8_DECODER(
        smoke_ids[:1], int8_memory, smoke_mask[:1]
    )
    assert tuple(fp32_single_logits.shape) == (1, 31, 10080)
    assert tuple(int8_single_logits.shape) == (1, 31, 10080)
    assert torch.isfinite(fp32_single_logits).all()
    assert torch.isfinite(int8_single_logits).all()

write_json(CONFIG / "int8_full_validation_quality_contract.json", {
    "schema_version": "hybrid_kd_int8_full_validation_quality_contract_v1",
    "created_at_utc": now(),
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "strategy": READY["strategy"],
    "runtime": {
        "onnxruntime": ort.__version__,
        "provider": "CPUExecutionProvider",
        "intra_op_threads": 1,
        "inter_op_threads": 1,
        "execution_mode": "ORT_SEQUENTIAL",
    },
    "fp32": {
        "encoder_sha256": sha256(FP32_ENCODER_PATH),
        "decoder_sha256": sha256(FP32_DECODER_PATH),
    },
    "int8_candidate": {
        "encoder_sha256": sha256(INT8_ENCODER_PATH),
        "decoder_sha256": sha256(INT8_DECODER_PATH),
    },
    "decoding": {
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_steps": 31,
        "batched_decoder_supported": BATCHED_DECODER_SUPPORTED,
        "batched_decoder_error": BATCHED_DECODER_ERROR,
    },
    "quality_gates": QUALITY_GATES,
    "selection_split": "Karpathy Validation",
    "validation_images": 5000,
    "validation_references": 25010,
    "physical_android_benchmark": False,
    "test_set_status": "sealed",
    "test_data_read_this_phase": False,
    "new_test_opening": False,
})
write_json(AUDIT / "cell03_runtime_smoke.json", {
    "schema_version": "hybrid_kd_int8_full_quality_runtime_smoke_v1",
    "created_at_utc": now(),
    "status": "passed",
    "ort_version": ort.__version__,
    "fp32_encoder_shape": list(fp32_memory.shape),
    "int8_encoder_shape": list(int8_memory.shape),
    "batched_decoder_supported": BATCHED_DECODER_SUPPORTED,
    "batched_decoder_error": BATCHED_DECODER_ERROR,
    "test_set_status": "sealed",
    "test_data_read_this_phase": False,
    "new_test_opening": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("CELL 3 — ONNX RUNTIME SMOKE")
print("Status                  : PASSED")
print("ONNX Runtime            :", ort.__version__)
print("Execution provider      : CPUExecutionProvider")
print("Fixed decoder batch     :", "BATCHED" if BATCHED_DECODER_SUPPORTED else "SINGLE (EXPECTED)")
print("FP32 / INT8 reload      : PASSED / PASSED")
print("Physical Android        : NOT BENCHMARKED")
print("Held-out Test           : SEALED")
print("Progress                : 3 / 6 code cells")

CELL 3 — ONNX RUNTIME SMOKE
Status                  : PASSED
ONNX Runtime            : 1.28.0
Execution provider      : CPUExecutionProvider
Fixed decoder batch     : BATCHED
FP32 / INT8 reload      : PASSED / PASSED
Physical Android        : NOT BENCHMARKED
Held-out Test           : SEALED
Progress                : 3 / 6 code cells


## Cell 4 — Paired FP32/INT8 generation, CE và exact resume

In [4]:
STARTED = time.perf_counter()
PRED_PATH = PREDICTION / "hybrid_kd_seed2026_fp32_int8_paired_predictions.jsonl"
SESSION_BUDGET_SECONDS = float(os.environ.get(
    "KLCN_SESSION_BUDGET_SECONDS", str(10.0 * 3600)
))

def read_jsonl(path):
    rows = {}
    if not Path(path).is_file():
        return rows
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                row = json.loads(line)
                image_id = int(row["image_id"])
                assert image_id not in rows
                rows[image_id] = row
    return rows

def append_jsonl(path, row):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

# Một JSONL giao dịch chứa cả FP32 và INT8 cho từng ảnh. Vì mỗi dòng chỉ
# được append sau khi hai nhánh hoàn tất, resume không thể lệch hai model.
if not PRED_PATH.exists():
    matches = sorted(NOTEBOOK_ROOT.rglob(PRED_PATH.name))
    assert len(matches) <= 1, matches
    if matches:
        prior_prediction = matches[0].resolve()
        prior_root = prior_prediction.parent.parent
        prior_contract_path = (
            prior_root / "configs/int8_full_validation_quality_contract.json"
        )
        prior_state_path = prior_root / "results/int8_full_validation_run_state.json"
        assert prior_contract_path.is_file() and prior_state_path.is_file()
        prior_contract = read_json(prior_contract_path)
        prior_state = read_json(prior_state_path)
        assert prior_contract["pair"] == PAIR
        assert prior_contract["student_id"] == STUDENT_ID
        assert int(prior_contract["seed"]) == SEED
        assert prior_contract["strategy"] == READY["strategy"]
        assert prior_contract["fp32"] == {
            "encoder_sha256": sha256(FP32_ENCODER_PATH),
            "decoder_sha256": sha256(FP32_DECODER_PATH),
        }
        assert prior_contract["int8_candidate"] == {
            "encoder_sha256": sha256(INT8_ENCODER_PATH),
            "decoder_sha256": sha256(INT8_DECODER_PATH),
        }
        assert prior_contract["quality_gates"] == QUALITY_GATES
        assert prior_state["pair"] == PAIR
        assert prior_state["paired_prediction_sha256"] == sha256(
            prior_prediction
        )
        assert prior_state["test_set_status"] == "sealed"
        assert prior_state["test_data_read_this_phase"] is False
        assert prior_state["new_test_opening"] is False
        shutil.copy2(prior_prediction, PRED_PATH)

paired_rows = read_jsonl(PRED_PATH)
assert set(paired_rows).issubset(set(VALIDATION_IMAGE_IDS))
fp32_rows = {
    image_id: row["fp32"] for image_id, row in paired_rows.items()
}
int8_rows = {
    image_id: row["int8"] for image_id, row in paired_rows.items()
}

def beam_generate(encoder, decoder, image):
    started_ns = time.perf_counter_ns()
    with torch.inference_mode():
        memory = encoder(image)
        beams = [([1], 0.0, False)]
        for _ in range(31):
            active_indices = [
                index for index, beam in enumerate(beams) if not beam[2]
            ]
            if not active_indices:
                break
            active = [beams[index] for index in active_indices]
            decoder_ids = torch.zeros(
                (len(active), 31), dtype=torch.long
            )
            decoder_mask = torch.zeros(
                (len(active), 31), dtype=torch.bool
            )
            for row_index, (tokens, _, _) in enumerate(active):
                length = min(len(tokens), 31)
                decoder_ids[row_index, :length] = torch.tensor(
                    tokens[:length], dtype=torch.long
                )
                decoder_mask[row_index, :length] = True
            expanded_memory = memory.expand(
                len(active), -1, -1
            ).contiguous()
            if BATCHED_DECODER_SUPPORTED:
                logits = decoder(
                    decoder_ids, expanded_memory, decoder_mask
                )
            else:
                logits = torch.cat([
                    decoder(
                        decoder_ids[index:index + 1],
                        memory,
                        decoder_mask[index:index + 1],
                    )
                    for index in range(len(active))
                ], dim=0)
            candidates = [
                beam for beam in beams if beam[2]
            ]
            for row_index, (tokens, score, _) in enumerate(active):
                position = min(len(tokens) - 1, 30)
                log_probs = F.log_softmax(
                    logits[row_index, position].float(), dim=-1
                )
                log_probs[[0, 1, 3]] = -torch.inf
                lexical_count = max(0, len(tokens) - 1)
                if lexical_count < 2:
                    log_probs[2] = -torch.inf
                values, indices = torch.topk(log_probs, 3)
                for value, token_id in zip(values.tolist(), indices.tolist()):
                    candidates.append((
                        tokens + [int(token_id)],
                        float(score + value),
                        int(token_id) == 2,
                    ))
            def rank(item):
                length = max(1, len(item[0]) - 1)
                return item[1] / (length ** 0.7)
            beams = sorted(candidates, key=rank, reverse=True)[:3]
            if all(item[2] for item in beams):
                break
        best = max(
            beams,
            key=lambda item: item[1] / (
                max(1, len(item[0]) - 1) ** 0.7
            ),
        )
    latency_ms = (time.perf_counter_ns() - started_ns) / 1e6
    return best[0], best[1], memory, latency_ms

def ce_for_image(decoder, memory, token_rows):
    sequences = [tokens for _, tokens in token_rows]
    batch = torch.zeros((len(sequences), 32), dtype=torch.long)
    for index, tokens in enumerate(sequences):
        length = min(len(tokens), 32)
        batch[index, :length] = torch.tensor(
            tokens[:length], dtype=torch.long
        )
    decoder_ids = batch[:, :-1]
    labels = batch[:, 1:]
    mask = decoder_ids.ne(0)
    with torch.inference_mode():
        if BATCHED_DECODER_SUPPORTED:
            logits = decoder(
                decoder_ids,
                memory.expand(
                    len(sequences), -1, -1
                ).contiguous(),
                mask,
            )
            loss_sum = F.cross_entropy(
                logits.float().reshape(-1, 10080),
                labels.reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
        else:
            loss_sum = torch.zeros((), dtype=torch.float32)
            for index in range(len(sequences)):
                logits = decoder(
                    decoder_ids[index:index + 1],
                    memory,
                    mask[index:index + 1],
                )
                loss_sum = loss_sum + F.cross_entropy(
                    logits.float().reshape(-1, 10080),
                    labels[index:index + 1].reshape(-1),
                    ignore_index=0,
                    reduction="sum",
                )
    token_count = int(labels.ne(0).sum())
    return float(loss_sum), token_count

completed = set(paired_rows)
INTERRUPTED_SAFELY = False
run_started = time.perf_counter()
starting_count = len(completed)
print(
    f"{PAIR} | ORT FP32 vs INT8 generation begins | "
    f"resume={starting_count}/5000 | budget={SESSION_BUDGET_SECONDS/3600:.2f} h",
    flush=True,
)
for ordinal, image_id in enumerate(VALIDATION_IMAGE_IDS):
    if image_id in completed:
        continue
    with Image.open(VAL_ROOT / filename(image_id)) as image_handle:
        image = VALIDATION_TRANSFORM(
            image_handle.convert("RGB")
        ).unsqueeze(0)

    fp32_tokens, fp32_score, fp32_memory, fp32_latency = beam_generate(
        FP32_ENCODER, FP32_DECODER, image
    )
    int8_tokens, int8_score, int8_memory, int8_latency = beam_generate(
        INT8_ENCODER, INT8_DECODER, image
    )
    fp32_ce_sum, fp32_ce_tokens = ce_for_image(
        FP32_DECODER, fp32_memory, VALIDATION_TOKEN_ROWS[image_id]
    )
    int8_ce_sum, int8_ce_tokens = ce_for_image(
        INT8_DECODER, int8_memory, VALIDATION_TOKEN_ROWS[image_id]
    )
    assert fp32_ce_tokens == int8_ce_tokens > 0
    fp32_row = {
        "image_id": image_id,
        "ordinal": ordinal,
        "caption": decode_tokens(fp32_tokens),
        "token_ids": fp32_tokens,
        "beam_score": fp32_score,
        "latency_ms": fp32_latency,
        "ce_loss_sum": fp32_ce_sum,
        "ce_tokens": fp32_ce_tokens,
    }
    int8_row = {
        "image_id": image_id,
        "ordinal": ordinal,
        "caption": decode_tokens(int8_tokens),
        "token_ids": int8_tokens,
        "beam_score": int8_score,
        "latency_ms": int8_latency,
        "ce_loss_sum": int8_ce_sum,
        "ce_tokens": int8_ce_tokens,
    }
    paired_row = {
        "image_id": image_id,
        "ordinal": ordinal,
        "fp32": fp32_row,
        "int8": int8_row,
    }
    append_jsonl(PRED_PATH, paired_row)
    paired_rows[image_id] = paired_row
    fp32_rows[image_id] = fp32_row
    int8_rows[image_id] = int8_row
    completed.add(image_id)

    session_completed = len(completed) - starting_count
    if session_completed <= 5 or len(completed) % 10 == 0:
        elapsed = max(time.perf_counter() - run_started, 1e-9)
        seconds_per_image = elapsed / max(session_completed, 1)
        remaining_hours = (
            (5000 - len(completed)) * seconds_per_image / 3600
        )
        print(
            f"{PAIR} | completed={len(completed):04d}/5000 | "
            f"fp32={fp32_latency:.1f} ms | int8={int8_latency:.1f} ms | "
            f"session_rate={seconds_per_image:.1f} s/image | "
            f"rough_remaining={remaining_hours:.1f} h",
            flush=True,
        )
    if time.perf_counter() - run_started >= SESSION_BUDGET_SECONDS:
        INTERRUPTED_SAFELY = True
        print("Session budget reached at image boundary:", len(completed))
        break

GENERATION_COMPLETE = len(completed) == 5000
write_json(RESULT / "int8_full_validation_run_state.json", {
    "schema_version": "hybrid_kd_int8_full_validation_run_state_v1",
    "created_at_utc": now(),
    "pair": PAIR,
    "status": "complete" if GENERATION_COMPLETE else "resume_ready",
    "completed_images": len(completed),
    "total_images": 5000,
    "interrupted_safely": INTERRUPTED_SAFELY,
    "paired_prediction_sha256": sha256(PRED_PATH),
    "test_set_status": "sealed",
    "test_data_read_this_phase": False,
    "new_test_opening": False,
})

print("CELL 4 — GENERATION")
print("Status                  :", "COMPLETE" if GENERATION_COMPLETE else "RESUME READY")
print("Completed images        :", len(completed), "/ 5000")
print("FP32 / INT8 rows        :", len(fp32_rows), "/", len(int8_rows))
print("Session interrupted     :", INTERRUPTED_SAFELY)
print("Held-out Test           : SEALED")
print("Progress                : 4 / 6 code cells")

HybridKD-S1 | ORT FP32 vs INT8 generation begins | resume=0/5000 | budget=10.00 h
HybridKD-S1 | completed=0001/5000 | fp32=385.4 ms | int8=264.3 ms | session_rate=0.8 s/image | rough_remaining=1.1 h
HybridKD-S1 | completed=0002/5000 | fp32=278.2 ms | int8=234.3 ms | session_rate=0.9 s/image | rough_remaining=1.2 h
HybridKD-S1 | completed=0003/5000 | fp32=329.0 ms | int8=233.9 ms | session_rate=0.9 s/image | rough_remaining=1.3 h
HybridKD-S1 | completed=0004/5000 | fp32=328.2 ms | int8=231.8 ms | session_rate=0.9 s/image | rough_remaining=1.3 h
HybridKD-S1 | completed=0005/5000 | fp32=298.3 ms | int8=217.7 ms | session_rate=0.9 s/image | rough_remaining=1.3 h
HybridKD-S1 | completed=0010/5000 | fp32=291.7 ms | int8=216.3 ms | session_rate=0.9 s/image | rough_remaining=1.3 h
HybridKD-S1 | completed=0020/5000 | fp32=268.0 ms | int8=193.2 ms | session_rate=0.9 s/image | rough_remaining=1.3 h
HybridKD-S1 | completed=0030/5000 | fp32=353.7 ms | int8=245.2 ms | session_rate=1.0 s/image | roug

## Cell 5 — COCO metrics, CE và quyết định quality gate

In [5]:
STARTED = time.perf_counter()
QUALITY_RESULT = None
if GENERATION_COMPLETE:
    assert set(fp32_rows) == set(int8_rows) == set(VALIDATION_IMAGE_IDS)
    fp32_hypotheses = {
        image_id: fp32_rows[image_id]["caption"]
        for image_id in VALIDATION_IMAGE_IDS
    }
    int8_hypotheses = {
        image_id: int8_rows[image_id]["caption"]
        for image_id in VALIDATION_IMAGE_IDS
    }
    fp32_metrics = compute_coco_metrics(
        VALIDATION_REFERENCES, fp32_hypotheses
    )
    int8_metrics = compute_coco_metrics(
        VALIDATION_REFERENCES, int8_hypotheses
    )
    fp32_ce_tokens = sum(row["ce_tokens"] for row in fp32_rows.values())
    int8_ce_tokens = sum(row["ce_tokens"] for row in int8_rows.values())
    assert fp32_ce_tokens == int8_ce_tokens
    fp32_ce = (
        sum(row["ce_loss_sum"] for row in fp32_rows.values())
        / fp32_ce_tokens
    )
    int8_ce = (
        sum(row["ce_loss_sum"] for row in int8_rows.values())
        / int8_ce_tokens
    )
    delta = {
        key: int8_metrics[key] - fp32_metrics[key]
        for key in fp32_metrics
    }
    delta["CE"] = int8_ce - fp32_ce
    gates = {
        "CIDEr-D": delta["CIDEr-D"] >= QUALITY_GATES["CIDEr-D_minimum_delta"],
        "BLEU-4": delta["BLEU-4"] >= QUALITY_GATES["BLEU-4_minimum_delta"],
        "CE": delta["CE"] <= QUALITY_GATES["CE_maximum_delta"],
        "finite": all(
            math.isfinite(value)
            for value in [
                *fp32_metrics.values(), *int8_metrics.values(),
                fp32_ce, int8_ce,
            ]
        ),
    }
    exact_caption_fraction = sum(
        fp32_hypotheses[image_id] == int8_hypotheses[image_id]
        for image_id in VALIDATION_IMAGE_IDS
    ) / 5000
    quality_passed = all(gates.values())
    QUALITY_RESULT = {
        "schema_version": "hybrid_kd_int8_full_validation_quality_result_v1",
        "created_at_utc": now(),
        "pair": PAIR,
        "student_id": STUDENT_ID,
        "seed": SEED,
        "strategy": READY["strategy"],
        "selection_split": "Karpathy Validation",
        "validation_images": 5000,
        "validation_references": 25010,
        "fp32": {**fp32_metrics, "CE": fp32_ce},
        "int8": {**int8_metrics, "CE": int8_ce},
        "delta_int8_minus_fp32": delta,
        "gates": gates,
        "quality_gate_passed": quality_passed,
        "deployment_decision": (
            "authorize_dynamic_int8_for_android_benchmark"
            if quality_passed else "retain_fp32_for_android_benchmark"
        ),
        "exact_caption_agreement_fraction": exact_caption_fraction,
        "test_set_status": "sealed",
        "test_data_read_this_phase": False,
        "new_test_opening": False,
    }
    write_json(RESULT / "int8_full_validation_quality_result.json", QUALITY_RESULT)
    print("FP32 metrics            :", QUALITY_RESULT["fp32"])
    print("INT8 metrics            :", QUALITY_RESULT["int8"])
    print("Delta INT8-FP32         :", delta)
    print("Quality gate            :", "PASSED" if quality_passed else "FAILED")

write_json(AUDIT / "cell05_quality_decision.json", {
    "schema_version": "hybrid_kd_int8_full_quality_decision_v1",
    "created_at_utc": now(),
    "status": "passed" if GENERATION_COMPLETE else "deferred_resume_required",
    "generation_complete": GENERATION_COMPLETE,
    "quality_result_available": QUALITY_RESULT is not None,
    "quality_gate_passed": (
        QUALITY_RESULT["quality_gate_passed"]
        if QUALITY_RESULT is not None else None
    ),
    "test_set_status": "sealed",
    "test_data_read_this_phase": False,
        "new_test_opening": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("CELL 5 — QUALITY")
print("Status                  :", "EVALUATED" if QUALITY_RESULT else "DEFERRED")
print("Selection split         : VALIDATION")
print("Held-out Test           : SEALED")
print("Progress                : 5 / 6 code cells")

PTBTokenizer tokenized 307821 tokens at 562151.93 tokens per second.
PTBTokenizer tokenized 52314 tokens at 203211.39 tokens per second.


{'testlen': 47315, 'reflen': 47339, 'guess': [47315, 42315, 37315, 32315], 'correct': [35954, 20088, 9883, 4689]}
ratio: 0.9994930184414331


PTBTokenizer tokenized 307821 tokens at 452863.71 tokens per second.
PTBTokenizer tokenized 52295 tokens at 239667.86 tokens per second.


{'testlen': 47296, 'reflen': 47355, 'guess': [47296, 42296, 37296, 32296], 'correct': [35946, 20119, 9908, 4704]}
ratio: 0.9987540914369972
FP32 metrics            : {'BLEU-1': 0.7595005254914521, 'BLEU-4': 0.3429633759856556, 'METEOR': 0.2647197823731836, 'ROUGE-L': 0.553525344358216, 'CIDEr-D': 1.1015041608013585, 'CE': 2.743398339449396}
INT8 metrics            : {'BLEU-1': 0.7590744811386937, 'BLEU-4': 0.34348053138273166, 'METEOR': 0.26464619199488904, 'ROUGE-L': 0.553592052744133, 'CIDEr-D': 1.0994972970063368, 'CE': 2.7427356072995472}
Delta INT8-FP32         : {'BLEU-1': -0.00042604435275839236, 'BLEU-4': 0.0005171553970760301, 'METEOR': -7.35903782945746e-05, 'ROUGE-L': 6.670838591704698e-05, 'CIDEr-D': -0.002006863795021685, 'CE': -0.0006627321498489103}
Quality gate            : PASSED
CELL 5 — QUALITY
Status                  : EVALUATED
Selection split         : VALIDATION
Held-out Test           : SEALED
Progress                : 5 / 6 code cells


## Cell 6 — Hoàn tất package, manifest và authorization bước Android

In [6]:
STARTED = time.perf_counter()
deployment_authorization_path = None
if GENERATION_COMPLETE:
    assert QUALITY_RESULT is not None
    selected_precision = "INT8_DYNAMIC" if QUALITY_RESULT["quality_gate_passed"] else "FP32"
    selected_roles = (
        ("encoder_candidate", "decoder_candidate")
        if QUALITY_RESULT["quality_gate_passed"]
        else ("encoder_fp32", "decoder_fp32")
    )
    deployment_authorization = {
        "schema_version": "hybrid_kd_android_benchmark_authorized_v1",
        "created_at_utc": now(),
        "status": "authorized",
        "scientific_role": "ablation",
        "product_deployment_override": True,
        "student_id": STUDENT_ID,
        "seed": SEED,
        "selected_precision": selected_precision,
        "quality_gate_passed": QUALITY_RESULT["quality_gate_passed"],
        "deployment_decision": QUALITY_RESULT["deployment_decision"],
        "quality_result": {
            "relative_path": "results/int8_full_validation_quality_result.json",
            "sha256": sha256(RESULT / "int8_full_validation_quality_result.json"),
        },
        "source_preflight_marker": {
            "name": PREFLIGHT_PATH.name,
            "sha256": sha256(PREFLIGHT_PATH),
        },
        "selected_artifacts": {
            "encoder": artifact_map[selected_roles[0]],
            "decoder": artifact_map[selected_roles[1]],
            "vocabulary": {
                "relative_path": "package/word_vocabulary_minfreq5.json",
                "size_bytes": VOCAB_PATH.stat().st_size,
                "sha256": sha256(VOCAB_PATH),
            },
            "deployment_contract": {
                "relative_path": "package/fp32_deployment_contract.json",
                "size_bytes": DEPLOYMENT_CONTRACT_PATH.stat().st_size,
                "sha256": sha256(DEPLOYMENT_CONTRACT_PATH),
            },
        },
        "required_next": "physical Android benchmark with warm-up and repeated latency/RAM measurements",
        "constraints": {
            "attach_preflight_output_for_models": True,
            "new_training": False,
            "checkpoint_reselection": False,
            "new_test_opening": False,
            "quality_threshold_change": False,
        },
        "prior_test_open_count": 0,
        "test_status": "sealed_not_used",
        "test_data_read_this_phase": False,
    }
    deployment_authorization_path = OUT / "HYB_ANDROID_BENCHMARK_AUTHORIZED.json"
    write_json(deployment_authorization_path, deployment_authorization)

manifest_rows = []
artifact_directories = [AUDIT, CONFIG, RESULT, PREDICTION]
for directory in artifact_directories:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })
if deployment_authorization_path is not None:
    manifest_rows.append({
        "relative_path": deployment_authorization_path.relative_to(OUT).as_posix(),
        "size_bytes": deployment_authorization_path.stat().st_size,
        "sha256": sha256(deployment_authorization_path),
    })
manifest_rows.sort(key=lambda row: row["relative_path"])

manifest_csv = MANIFEST / "int8_full_validation_quality_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["relative_path", "size_bytes", "sha256"])
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / "int8_full_validation_quality_manifest.json"
write_json(manifest_json, {
    "schema_version": "hybrid_kd_int8_full_validation_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(f"{row['sha256']}  {row['relative_path']}\n" for row in manifest_rows),
    encoding="utf-8",
)
for row in manifest_rows:
    path = OUT / row["relative_path"]
    assert path.is_file() and path.stat().st_size == int(row["size_bytes"])
    assert sha256(path) == row["sha256"]

marker = {
    "schema_version": (
        "hybrid_kd_int8_full_validation_quality_complete_v1"
        if GENERATION_COMPLETE
        else "hybrid_kd_int8_full_validation_quality_resume_ready_v1"
    ),
    "created_at_utc": now(),
    "status": "complete" if GENERATION_COMPLETE else "resume_ready",
    "student_id": STUDENT_ID,
    "seed": SEED,
    "candidate": READY["strategy"],
    "completed_images": len(completed),
    "validation_images": 5000,
    "validation_references": 25010,
    "quality_gate_passed": (
        QUALITY_RESULT["quality_gate_passed"] if QUALITY_RESULT is not None else None
    ),
    "deployment_decision": (
        QUALITY_RESULT["deployment_decision"] if QUALITY_RESULT is not None else "resume_required"
    ),
    "android_benchmark_authorized": GENERATION_COMPLETE,
    "caption_quality_evaluated": GENERATION_COMPLETE,
    "physical_android_benchmark_performed": False,
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "prior_test_open_count": 0,
    "new_test_opening": False,
    "test_data_read_this_phase": False,
    "test_status": "sealed_not_used",
    "manifest": {
        "csv": {
            "relative_path": manifest_csv.relative_to(OUT).as_posix(),
            "size_bytes": manifest_csv.stat().st_size,
            "sha256": sha256(manifest_csv),
            "rows": len(manifest_rows),
        },
        "json": {
            "relative_path": manifest_json.relative_to(OUT).as_posix(),
            "size_bytes": manifest_json.stat().st_size,
            "sha256": sha256(manifest_json),
        },
        "sha256sums": {
            "relative_path": sha_file.relative_to(OUT).as_posix(),
            "size_bytes": sha_file.stat().st_size,
            "sha256": sha256(sha_file),
        },
    },
    "manifest_rows": len(manifest_rows),
    "elapsed_seconds_cell06": time.perf_counter() - STARTED,
}
marker_name = (
    "HYB_INT8_FULL_VALIDATION_COMPLETE.json"
    if GENERATION_COMPLETE
    else "HYB_INT8_FULL_VALIDATION_RESUME_READY.json"
)
write_json(OUT / marker_name, marker)

print("INT8 FULL VALIDATION QUALITY — FINAL")
print("Status                  :", marker["status"].upper())
print("Student / seed          :", STUDENT_ID, "/", SEED)
print("Completed images        :", len(completed), "/ 5000")
print("Quality gate            :", marker["quality_gate_passed"])
print("Deployment decision     :", marker["deployment_decision"])
print("New training            : NOT PERFORMED")
print("Held-out Test reopened  : False")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Next                    :", "Android physical benchmark" if GENERATION_COMPLETE else "attach this output and resume")
print("Progress                : 6 / 6 code cells")

INT8 FULL VALIDATION QUALITY — FINAL
Status                  : COMPLETE
Student / seed          : S1 / 2026
Completed images        : 5000 / 5000
Quality gate            : True
Deployment decision     : authorize_dynamic_int8_for_android_benchmark
New training            : NOT PERFORMED
Held-out Test reopened  : False
Manifest                : 10 / 10 VERIFIED
Next                    : Android physical benchmark
Progress                : 6 / 6 code cells
